## Creacion de Nuevas Columnas

In [71]:
import pandas as pd
# missingno para visualizar datos faltantes en df
import missingno as msno
# numpy para manejar datos específicos con pandas (procesamiento computacional más rápido)
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt


In [72]:
## Carga del Data set Limpio
df_autos = pd.read_csv('/home/javif/UTA/Proyecto_Fabian/Proyecto_Autos_Usados/data/processed/vehicles_clean_imputed.csv')

In [73]:
df_autos.head()

,price,year,manufacturer,model,condition,cylinders,fuel,odometer,transmission,drive,type,state
0,33590.0,2014,gmc,sierra 1500 crew cab slt,good,8,gas,57923.0,other,4wd,pickup,al
1,22590.0,2010,chevrolet,silverado 1500,good,8,gas,71229.0,other,4wd,pickup,al
2,39590.0,2020,chevrolet,silverado 1500 crew,good,8,gas,19160.0,other,4wd,pickup,al
3,30990.0,2017,toyota,tundra double cab sr,good,8,gas,41124.0,other,4wd,pickup,al
4,15000.0,2013,ford,f-150 xlt,excellent,6,gas,128000.0,automatic,rwd,truck,al


In [74]:
df_autos.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249706 entries, 0 to 249705
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   price         249706 non-null  float64
 1   year          249706 non-null  int64  
 2   manufacturer  249706 non-null  object 
 3   model         249706 non-null  object 
 4   condition     249706 non-null  object 
 5   cylinders     249706 non-null  int64  
 6   fuel          249706 non-null  object 
 7   odometer      249706 non-null  float64
 8   transmission  249706 non-null  object 
 9   drive         249706 non-null  object 
 10  type          249706 non-null  object 
 11  state         249706 non-null  object 
dtypes: float64(2), int64(2), object(8)
memory usage: 22.9+ MB


In [75]:
df_autos.nunique()

price           14727
year               43
manufacturer       41
model           20921
condition           6
cylinders           9
fuel                5
odometer        93085
transmission        3
drive               3
type               13
state              51
dtype: int64

In [76]:
# --- 1. Estadísticos descriptivos de columnas numéricas ---
print("\n=== Estadísticos básicos (numéricos) ===")
print(df_autos.describe().T)    # media, std, min, max, quartiles

# --- 2. Varianza de columnas numéricas ---
print("\n=== Varianza ===")
print(df_autos.var(numeric_only=True))

# --- 3. Distribución de columnas categóricas ---
print("\n=== Frecuencias de columnas categóricas ===")
for col in df_autos.select_dtypes(include=['object']).columns:
    print(f"\n--- {col} ---")
    print(df_autos[col].value_counts().head(10))   # top 10 valores
    print("Nº categorías:", df_autos[col].nunique())

# --- 4. Correlación entre variables numéricas ---
print("\n=== Matriz de correlación ===")
corr = df_autos.corr(numeric_only=True)
print(corr)

# --- 5. Correlación específicamente con el precio ---
print("\n=== Correlación con precio ===")
print(corr['price'].sort_values(ascending=False))



=== Estadísticos básicos (numéricos) ===
              count          mean           std     min      25%      50%  \
price      249706.0  17964.362498  14532.374662   500.0   6995.0  14000.0   
year       249706.0   2011.548065      6.592961  1980.0   2008.0   2013.0   
cylinders  249706.0      5.829343      1.481179     3.0      4.0      6.0   
odometer   249706.0  96845.872094  62532.004071     0.0  42581.0  93235.5   

                75%       max  
price       25900.0  449500.0  
year         2017.0    2022.0  
cylinders       7.0      12.0  
odometer   140000.0  300000.0  

=== Varianza ===
price        2.111899e+08
year         4.346713e+01
cylinders    2.193891e+00
odometer     3.910252e+09
dtype: float64

=== Frecuencias de columnas categóricas ===

--- manufacturer ---
manufacturer
ford         41841
chevrolet    31877
toyota       21816
honda        14607
nissan       12242
jeep         11529
gmc           9742
ram           9494
bmw           8889
dodge         7832
Name:

####  Age del Vehículo (en años)


In [77]:
df_autos['age'] = 2025 - df_autos['year']

####  Condition Encoded (Ordinal: 0-5)

In [78]:
condition_mapping = {
    'salvage': 0,
    'fair': 1,
    'good': 2,
    'excellent': 3,
    'like new': 4,
    'new': 5
}
df_autos['condition_encoded'] = df_autos['condition'].map(condition_mapping)


#### Log del Odómetro

In [79]:
df_autos['log_odometer'] = np.log1p(df_autos['odometer'])

#### Mileage per Year

In [80]:
df_autos['mileage_per_year'] = df_autos['odometer'] / (df_autos['age'] + 1)

#### 5. CYLINDERS_SQUARED (relación no-lineal)

In [81]:
df_autos['cylinders_squared'] = df_autos['cylinders'] ** 2

#### Age × Condition

In [82]:
df_autos['age_x_condition'] = df_autos['age'] * df_autos['condition_encoded']

#### Age × Log(Odometer)

In [ ]:
df_autos['age_x_log_odometer'] = df_autos['age'] * df_autos['log_odometer']


#### Cilindros por condicion

In [98]:
df_autos['cylinders_x_condition'] = df_autos['cylinders'] * df_autos['condition_encoded']

#### Agrupar tipos de vehiculos en catogorias 

In [84]:
#def simplify_type(x):
#    if x in ["van", "mini-van"]:
#        return "van"
#    elif x in ["sedan", "suv", "pickup", "truck", "coupe", "hatchback"]:
#        return x
#    else:
#        return "other"

#df_autos["type_simple"] = df_autos["type"].apply(simplify_type)

#### gas 86% diesel 5.7% hybrid, electric, other → 8% pero muy dispersos

In [93]:
#def simplify_fuel(x):
#    if x in ["gas", "diesel"]:
#        return x
#    else:
#        return "other"

#df_autos["fuel_simple"] = df_autos["fuel"].apply(simplify_fuel)

#### Is Automatic Transmission

In [94]:
df_autos['is_automatic'] = (df_autos['transmission'] == 'automatic').astype(int)


####  Is Popular Drive Type (4WD es 44%, muy importante)

In [95]:
df_autos['is_4wd'] = (df_autos['drive'] == '4wd').astype(int)
df_autos['is_rwd'] = (df_autos['drive'] == 'rwd').astype(int)

#### TARGET MEAN ENCODING en Estados

In [88]:
#state_avg = df_autos.groupby("state")["price"].mean()
#df_autos["state_price_avg"] = df_autos["state"].map(state_avg)

#### TARGET MEAN ENCODING en precios

In [89]:
#brand_avg = df_autos.groupby("manufacturer")["price"].mean()
#df_autos["manufacturer_price_avg"] = df_autos["manufacturer"].map(brand_avg)

In [100]:
columns_final = [
    # Target
    'price',
    
    # Numéricas originales (SIN year)
    'cylinders',
    'odometer',
    
    # Categóricas originales (para OHE/encoding posterior)
    'manufacturer',
    'condition',
    'fuel',
    'transmission',
    'drive',
    'type',
    'state',
    
    # Features numéricas derivadas
    'age',
    'condition_encoded',
    'log_odometer',
    'mileage_per_year',
    'cylinders_squared',
    
    # Interacciones
    'age_x_condition',
    'age_x_log_odometer',
    'cylinders_x_condition',
    
    # Binarias (SOLO 2)
    'is_automatic',
    'is_4wd',
    'is_rwd',
]

df_final = df_autos[columns_final].copy()

print("="*80)
print("DATASET FINAL")
print("="*80)
print(f"Filas: {df_final.shape[0]:,}")
print(f"Columnas: {df_final.shape[1]}")
print(f"Valores nulos: {df_final.isnull().sum().sum()}")
print(f"Memoria: {df_final.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nCOLUMNAS:")
for i, col in enumerate(df_final.columns, 1):
    print(f"{i:2d}. {col:30s} {df_final[col].dtype}")

# ═══════════════════════════════════════════════════════════════════════════════
# GUARDAR
# ═══════════════════════════════════════════════════════════════════════════════

df_final.to_csv('../data/processed/vehicles_with_features.csv', index=False)
print(f"\n✓ Archivo guardado: ../data/processed/vehicles_with_features.csv")

print("\nPrimeras 5 filas:")
print(df_final.head())

DATASET FINAL
Filas: 249,706
Columnas: 21
Valores nulos: 0
Memoria: 116.50 MB

COLUMNAS:
 1. price                          float64
 2. cylinders                      int64
 3. odometer                       float64
 4. manufacturer                   object
 5. condition                      object
 6. fuel                           object
 7. transmission                   object
 8. drive                          object
 9. type                           object
10. state                          object
11. age                            int64
12. condition_encoded              int64
13. log_odometer                   float64
14. mileage_per_year               float64
15. cylinders_squared              int64
16. age_x_condition                int64
17. age_x_log_odometer             float64
18. cylinders_x_condition          int64
19. is_automatic                   int64
20. is_4wd                         int64
21. is_rwd                         int64

✓ Archivo guardado: ../data/proc